In [18]:
library(httr2)
library(jsonlite)
library(tidyverse)
library(janitor)
library(lubridate)


## Medicaid provider files

The provider files are searchable in two ways: First by service provider, then by billing provider. Both of these need to have at least 10 clients in any month they are published. 

RHA has several NPI's and we'll check all of them. 

* 1043379340 is the bear creek facility. Start with that, but then check for the Bear Creek NPI. 
* about


## Retrieve the NPI's for RHA

This uses the CMS API to get the results

In [19]:
## Import the NPI listing. 

api_url <- "https://npiregistry.cms.hhs.gov/api/?number=&enumeration_type=NPI-2&taxonomy_description=&name_purpose=&first_name=&use_first_name_alias=&last_name=&organization_name=RHA*&address_purpose=&city=&state=&postal_code=&country_code=&limit=1000&skip=&pretty=&version=2.1"

# Use fromJSON to fetch and parse the data in one step
# It returns a list, and we extract the 'results' data frame
rha_data <- jsonlite::fromJSON(api_url)$results |> 
  as_tibble() |> 
  unnest(basic)

# Inspect the result
glimpse(rha_data)

Rows: 200
Columns: 24
$ addresses                             <list> [<data.frame[2 x 10]>], [<data.frame[2 x 10]>], [<data.frame[2 x 9]>], [<data.f…
$ authorized_official_first_name        <chr> "LAETITIA", "ALLEYA", "MILLIE", "CONNIE", "MICHAEL", "MICHAEL", "BRANDICE", "JEN…
$ authorized_official_last_name         <chr> "ODUNZE", "KIM", "SIMS", "HEFNER", "SCHUSTER", "SCHUSTER", "WALTERS", "LOZANO", …
$ authorized_official_middle_name       <chr> "N", NA, NA, NA, NA, NA, NA, NA, "D", "D", "D", "D", "D", "D", "D", "D", "D", "D…
$ authorized_official_name_prefix       <chr> "Mrs.", NA, "--", "--", "Mr.", "Mr.", NA, "--", NA, NA, "--", "--", NA, NA, "--"…
$ authorized_official_name_suffix       <chr> "--", NA, "--", "--", "--", "--", NA, "--", NA, NA, "--", "--", NA, NA, "--", "-…
$ authorized_official_telephone_number  <chr> "8012252145", "4807881009", "8282481117", "8283945563", "4059170300", "405917030…
$ authorized_official_title_or_position <chr> "ADMINISTRATOR", "Owner", "Director"

In [7]:
# Convert API results to a tibble for easier data manipulation
rha_data <- tibble(result = api_result) |>
  mutate(
    npi = map_chr(result, ~ purrr::pluck(.x, "number", .default = NA_character_)),
    enumeration_type = map_chr(result, ~ purrr::pluck(.x, "enumeration_type", .default = NA_character_)),
    org_name = map_chr(result, ~ purrr::pluck(.x, "basic", "organization_name", .default = NA_character_)),
    first_name = map_chr(result, ~ purrr::pluck(.x, "basic", "first_name", .default = NA_character_)),
    last_name  = map_chr(result, ~ purrr::pluck(.x, "basic", "last_name", .default = NA_character_))
  )


There was a problem - the NPI API only searches by name using the primary name, which was, for example, Bear Creek. Instead, we need to download and process the full NPI file. 

## Process NPI download

In [20]:
npi_base <- rha_data |> 
  select(npi = number, enumeration_type, org_name = organization_name)


In [ ]:

# 2. Create a long table for "other names"
other_names_long <- rha_data |>
  select(npi = number, other_names) |>
  # Unnest the other_names data frames, keeping records that had no other names
  unnest(other_names, keep_empty = FALSE)

# 3. Create a long table for "addresses"
addresses_long <- rha_data |>
  select(npi = number, addresses) |>
  unnest(addresses, keep_empty = FALSE) 

# 4. Create a long table for "identifiers"
identifiers_long <- rha_data |>
  select(npi = number, identifiers) |>
  unnest(identifiers, keep_empty = TRUE) |> 
  filter_out( is.na( state))

This was a problem: The API only searches the primary name, not the DBA names, which excludes Bear Creek for example. Instead, we have to download and filter the large NPI provider list. 

(Downloaded to the `data` folder)

In [42]:
data_path <- "data/NPPES_Data_Dissemination_April_2026_V2"

header_fnames <- list.files(path = data_path, pattern = ".*header.csv", full.names = TRUE)
filenames <- str_replace(header_fnames, "_fileheader", "")  
headers <- list()

for (f in filenames) {
  header_name <- str_replace(f, ".csv", "_fileheader.csv") 
  raw_data <- read_csv(header_name, col_types = cols(.default = "c"), show_col_types = FALSE) |> clean_names()  |>  colnames()  
  headers[[f]] <- raw_data

}

print (headers)



$`data/NPPES_Data_Dissemination_April_2026_V2/endpoint_pfile_20050523-20260412.csv`
 [1] "npi"                             "endpoint_type"                   "endpoint_type_description"      
 [4] "endpoint"                        "affiliation"                     "endpoint_description"           
 [7] "affiliation_legal_business_name" "use_code"                        "use_description"                
[10] "other_use_description"           "content_type"                    "content_description"            
[13] "other_content_description"       "affiliation_address_line_one"    "affiliation_address_line_two"   
[16] "affiliation_address_city"        "affiliation_address_state"       "affiliation_address_country"    
[19] "affiliation_address_postal_code"

$`data/NPPES_Data_Dissemination_April_2026_V2/npidata_pfile_20050523-20260412.csv`
  [1] "npi"                                                                    
  [2] "entity_type_code"                                               

In [52]:
all_data <- list()

for (n in names(headers) ) {
  var_names <- unlist ( headers[n] )
  shortname <- str_match(n, "/(.*?)_pfile.csv")[,2]
  
 # data <- read_csv(n, col_types = cols(.default = "c"), show_col_types = FALSE, col_names = var_names)
 # all_data[[shortname]] <- data
}

In [ ]:


# Read the header file once
file_headers <- read_csv(
  paste0(data_path, "fileheader.csv"), 
  show_col_types = FALSE
)



In [ ]:



# 1. Get the column names specifically for the main NPI data file
npidata_cols <- file_headers |>
  filter(`File Name` == "npidata") |>
  pull(`Column Name`)

# --- IMPORTANT ---
# 2. Update this filename to match the one you downloaded.
npidata_filename <- "npidata_pfile_20050523-20240407.csv"

# 3. Read the main NPI data file
npi_data <- read_csv(
  paste0(data_path, npidata_filename),
  col_names = npidata_cols,
  col_types = cols(.default = "c"), # Read all columns as character for speed/safety
  show_col_types = FALSE
)

# Verify the import
glimpse(npi_data)